In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import torch

from src.utils.data_utils import center_crop_npy

os.chdir("..")
os.getcwd()

# Calculating from tiles

In [ ]:
mod = "tessera"
size = 128

paths = glob.glob(f"data/s2bms/eo/{mod}/*UKBMS*.npy")
rows = []
for p in paths:
    arr = np.load(p).transpose(2, 0, 1)
    if arr.dtype != np.dtype("float32"):
        arr = arr.astype(dtype="float32", copy=False)
    arr = center_crop_npy(arr, (64 if mod == "aef" else 128, size, size))
    if np.isinf(arr).any():
        arr[np.isinf(arr)] = np.nan
    tensor = torch.from_numpy(arr)
    row = {f"emb_{i}": v.item() for i, v in enumerate(tensor.nanmean(dim=(-2, -1)))}
    row["name_loc"] = p.split("/")[-1].split(".")[0].replace(f"{mod}_", "")
    rows.append(row)

In [ ]:
df = pd.DataFrame(rows)
df

In [ ]:
if not os.path.exists(f"data/s2bms/eo/avr_{mod}_{size}.csv"):
    df.to_csv(f"data/s2bms/eo/avr_{mod}_{size}.csv", index=False)

# Merging-in unlabelled

In [ ]:
size = 128
mod = "aef"

df = pd.read_csv(f"data/s2bms/eo/avr_{mod}_{size}.csv")
df_unlabelled = pd.read_csv(f"data/s2bms/eo/avr_{mod}_{size}_just_unlabelled.csv")

df_merged = pd.concat([df, df_unlabelled])
df_merged.name_loc

if not os.path.exists(f"data/s2bms/eo/avr_{mod}_{size}_unlabelled.csv"):
    df.to_csv(f"data/s2bms/eo/avr_{mod}_{size}_unlabelled.csv")
else:
    print("Already saved")